In [ ]:
import json
import os
import torch
from torch.nn import functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from typing import List, Dict
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

MODEL_PATH = ""
DATASET_PATH = ''
OUTPUT_DIR = ''
OUTPUT_FILENAME = 'medexqa_greedy_decoding_qwen_results.jsonl' 

BATCH_SIZE = 4
MAX_NEW_TOKENS = 20
ALPHA = 0.5

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

model.eval()
device = next(model.parameters()).device

def load_dataset(path: str) -> List[Dict]:
    dataset = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                dataset.append(json.loads(line))
    return dataset

def standard_greedy_batch(input_ids, attention_mask):
    output_ids = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        eos_token_id=tokenizer.eos_token_id
    )
    return tokenizer.batch_decode(output_ids[:, input_ids.shape[1]:], skip_special_tokens=True)

def context_aware_greedy_batch(
    input_ids_with_context: torch.Tensor,
    attention_mask_with_context: torch.Tensor,
    input_ids_without_context: torch.Tensor,
    attention_mask_without_context: torch.Tensor
) -> List[str]:

    batch_size = input_ids_with_context.shape[0]

    with torch.no_grad():
        full_outputs = model(input_ids_with_context, attention_mask=attention_mask_with_context, use_cache=True)
        full_past_key_values = full_outputs.past_key_values
        full_logits = full_outputs.logits[:, -1, :]

        q_only_outputs = model(input_ids_without_context, attention_mask=attention_mask_without_context, use_cache=True)
        q_only_past_key_values = q_only_outputs.past_key_values
        q_only_logits = q_only_outputs.logits[:, -1, :]

    adjusted_logits = (1 + ALPHA) * full_logits - ALPHA * q_only_logits
    next_token = torch.argmax(adjusted_logits, dim=-1, keepdim=True)

    generated_sequences = [next_token]
    unfinished_sequences = torch.ones(batch_size, dtype=torch.long, device=device)

    for _ in range(1, MAX_NEW_TOKENS):
        if unfinished_sequences.max() == 0:
            break

        attention_mask_with_context = torch.cat([
            attention_mask_with_context, 
            torch.ones((batch_size, 1), dtype=torch.long, device=device)
        ], dim=-1)
        
        attention_mask_without_context = torch.cat([
            attention_mask_without_context,
            torch.ones((batch_size, 1), dtype=torch.long, device=device)
        ], dim=-1)

        with torch.no_grad():
            full_outputs = model(next_token, attention_mask=attention_mask_with_context, past_key_values=full_past_key_values, use_cache=True)
            full_logits = full_outputs.logits[:, -1, :]
            full_past_key_values = full_outputs.past_key_values

            q_only_outputs = model(next_token, attention_mask=attention_mask_without_context, past_key_values=q_only_past_key_values, use_cache=True)
            q_only_logits = q_only_outputs.logits[:, -1, :]
            q_only_past_key_values = q_only_outputs.past_key_values
        
        adjusted_logits = (1 + ALPHA) * full_logits - ALPHA * q_only_logits
        next_token_choice = torch.argmax(adjusted_logits, dim=-1, keepdim=True)

        next_token = torch.where(unfinished_sequences.unsqueeze(-1) == 1, next_token_choice, torch.tensor(tokenizer.pad_token_id, device=device))
        generated_sequences.append(next_token)

        unfinished_sequences = unfinished_sequences & (next_token_choice.squeeze(-1) != tokenizer.eos_token_id)

    final_tokens = torch.cat(generated_sequences, dim=-1)
    return tokenizer.batch_decode(final_tokens, skip_special_tokens=True)

def main():
    medmcqa_dataset = load_dataset(DATASET_PATH)
    results = []

    for i in tqdm(range(0, len(medmcqa_dataset), BATCH_SIZE)):
        batch_data = medmcqa_dataset[i:i + BATCH_SIZE]
        
        full_prompts, questions_only_prompts, original_questions, contexts = [], [], [], []

        for data in batch_data:
            docs = data.get('docs', [])
            context_str = "\n".join([f"[Document{idx+1}] {doc.get('text', 'No text')}" for idx, doc in enumerate(docs)])
            question_text = data.get('raw_question', '')
            
            original_questions.append(question_text)
            contexts.append(context_str)

            messages_full = [
                {"role": "system", "content": "You are a helpful assistant. Use the provided documents to answer the multiple-choice question. Your answer must be only the letter of the correct option (e.g., A, B, C, or D)."},
                {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion:\n{question_text}\n\nAnswer:"}
            ]
            full_prompts.append(tokenizer.apply_chat_template(messages_full, tokenize=False, add_generation_prompt=True))

            messages_q_only = [
                {"role": "system", "content": "You are a helpful assistant. Please answer the multiple-choice question. Your answer must be only the letter of the correct option (e.g., A, B, C, or D)."},
                {"role": "user", "content": f"Question:\n{question_text}\n\nAnswer:"}
            ]
            questions_only_prompts.append(tokenizer.apply_chat_template(messages_q_only, tokenize=False, add_generation_prompt=True))

        tokenizer.padding_side = 'left'
        inputs_full_dict = tokenizer(full_prompts, return_tensors="pt", padding='longest', truncation=True, max_length=7000).to(device)
        inputs_q_only_dict = tokenizer(questions_only_prompts, return_tensors="pt", padding='longest', truncation=True, max_length=7000).to(device)

        standard_outputs = standard_greedy_batch(inputs_full_dict.input_ids, inputs_full_dict.attention_mask)
        context_aware_outputs = context_aware_greedy_batch(
            inputs_full_dict.input_ids, inputs_full_dict.attention_mask,
            inputs_q_only_dict.input_ids, inputs_q_only_dict.attention_mask
        )

        for j in range(len(batch_data)):
            result = {
                'index': i + j,
                'question': original_questions[j],
                'context': contexts[j],
                'standard_output': standard_outputs[j],
                'context_aware_ou